# Security Attack Test Suite (RQ2: in-transit / MITM tampering)

Evaluates the **revised** ThermoCAPTCHA protocol implemented in `thermocaptcha_protocol.py`. This is the same module `Final_code.ipynb` uses:
server-issued challenge/session, client signature over $H(C\parallel sid\parallel H(I)\parallel pk_d)$, server-derived fingerprint, context-bound single-use token, and HMAC-authenticated website requests.

## Attack set (500 attempts each, 2,500 total)
| Attack | What the adversary does |
|---|---|
| Replay | Re-submits a captured, already-accepted capture payload verbatim |
| Stale timestamp | Submits a correctly signed capture for a challenge older than the 120 s TTL |
| Nonce reuse | Reuses a consumed challenge/nonce with a freshly computed signature (alternating: verbatim reused challenge, or forged challenge ID with reused nonce) |
| Modified binary (thermal image) | Flips image bytes in transit; signature unchanged |
| Modified metadata (challenge fields) | Changes `challenge.nonce` / `issued_at` in transit; signature unchanged |

A **valid control** (honest captures) confirms the server accepts legitimate traffic, so 0 accepted attacks is not caused by a broken pipeline. A supplementary section tests the new website-to-server channel (it is not part of the 2,500).

## Setup

In [1]:
import json, os, shutil, time, base64, random, collections, uuid
import pandas as pd
import thermocaptcha_protocol as tp

SOURCE_DB = "thermal_captcha.db"
EVAL_DB = "rq2_mitm_eval.db"          # fresh copy per run -> reproducible, does not touch the shipped DB
TEST_IMAGE_PATH = "./25.bmp"
SITE_KEY = "28024d2408754883a7eff1db9cea2b00"
WEBSITE_NAME = "microsoft.com"
random.seed(2026)

shutil.copy(SOURCE_DB, EVAL_DB)
tp.ensure_demo_secrets(SITE_KEY)
server = tp.ThermoCaptchaServer(EVAL_DB)                      # real thermal detector (thermalImageChecker)
website = tp.Website(SITE_KEY, WEBSITE_NAME, tp.load_website_secret())
with open(TEST_IMAGE_PATH, "rb") as f:
    IMAGE = f.read()

def new_device():
    # Simulated devices use in-memory persistent keys (one key per device, reused across challenges).
    return tp.ThermoCaptchaClient(tp.rsa.generate_private_key(public_exponent=65537, key_size=2048))

print("Protocol:", tp.PROTOCOL_VERSION, "| eval DB:", EVAL_DB)

NUM_ATTEMPTS = 500
NUM_CONTROL = 100
OBS_IP, OBS_UA = "127.0.0.1", "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
victim = new_device()          # honest user's persistent device key

def honest_capture(now=None):
    ch = server.issue_challenge(SITE_KEY, now=now)
    return ch, victim.build_capture_payload(ch, IMAGE)

def run_attack(name, attempt_fn, n=NUM_ATTEMPTS):
    accepted, reasons = 0, collections.Counter()
    t0 = time.time()
    for i in range(n):
        res = attempt_fn(i)
        accepted += int(res.ok)
        reasons[res.reason] += 1
    print(f"{name:<22} attempts={n} accepted={accepted} ({time.time()-t0:.1f}s)  reasons={dict(reasons)}")
    return {"Attack Type": name, "Attempts": n, "Accepted": accepted, "Rejected": n - accepted,
            "Success Rate (%)": 100.0 * accepted / n, "Rejection reasons": dict(reasons)}

NOTE: generated ephemeral demo secrets for: THERMOCAPTCHA_SERVER_SECRET, THERMOCAPTCHA_WEBSITE_SECRET, THERMOCAPTCHA_SITE_SECRETS[28024d2408754883a7eff1db9cea2b00] (set them in the environment for a real deployment).
Protocol: thermocaptcha-v2 | eval DB: rq2_mitm_eval.db


## Valid control (honest captures must be accepted)

In [2]:
control = run_attack("Valid control", lambda i: server.verify_capture(honest_capture()[1], OBS_IP, OBS_UA), NUM_CONTROL)
assert control["Accepted"] == NUM_CONTROL, 'pipeline rejects honest traffic; attack results would be meaningless'

Valid control          attempts=100 accepted=100 (0.5s)  reasons={'accepted': 100}


## Test 1: Replay attack (500 attempts)
Each attempt replays a distinct capture that the server has already accepted once (the first submission is not counted).

In [3]:
def replay_attempt(i):
    _, payload = honest_capture()
    assert server.verify_capture(payload, OBS_IP, OBS_UA).ok      # legitimate first use (not counted)
    return server.verify_capture(payload, OBS_IP, OBS_UA)         # replay (counted)

replay = run_attack("Replay", replay_attempt)

Replay                 attempts=500 accepted=0 (2.6s)  reasons={'challenge already used (replay)': 500}


## Test 2: Stale challenge attack (500 attempts)
The challenge was issued 130 s ago (> 120 s TTL) and is correctly signed. Freshness is judged from the server's stored `issued_at`, not from the client.

In [4]:
stale = run_attack("Stale timestamp", lambda i: server.verify_capture(
    honest_capture(now=time.time() - 130)[1], OBS_IP, OBS_UA))

Stale timestamp        attempts=500 accepted=0 (1.3s)  reasons={'challenge expired': 500}


## Test 3: Nonce reuse attack (500 attempts)
The adversary takes a challenge that has already been consumed and signs a new capture over it. Even attempts reuse the challenge verbatim; odd attempts forge a new `challenge_id` that carries the reused nonce.

In [5]:
def nonce_reuse_attempt(i):
    ch, payload = honest_capture()
    assert server.verify_capture(payload, OBS_IP, OBS_UA).ok      # consumes the challenge
    reused = json.loads(ch)
    if i % 2 == 1:
        reused["challenge"]["challenge_id"] = str(uuid.uuid4())
    return server.verify_capture(victim.build_capture_payload(json.dumps(reused), IMAGE), OBS_IP, OBS_UA)

nonce = run_attack("Nonce reuse", nonce_reuse_attempt)

Nonce reuse            attempts=500 accepted=0 (3.4s)  reasons={'challenge already used (replay)': 250, 'unknown challenge (not issued by server)': 250}


## Test 4a: Modified thermal image (500 attempts)
Bytes of the thermal image are flipped in transit; the signature is left unchanged.

In [6]:
def modified_image_attempt(i):
    _, payload = honest_capture()
    p = json.loads(payload)
    img = bytearray(base64.b64decode(p["thermal_image"]))
    for pos in (0, 10, 50, 100, random.randrange(len(img))):
        img[pos] ^= 0xFF
    p["thermal_image"] = base64.b64encode(bytes(img)).decode()
    return server.verify_capture(json.dumps(p), OBS_IP, OBS_UA)

modified_image = run_attack("Modified binary", modified_image_attempt)

Modified binary        attempts=500 accepted=0 (1.9s)  reasons={'invalid signature': 500}


## Test 4b: Modified challenge (500 attempts)
`challenge.nonce` and `challenge.issued_at` are altered in transit; the signature is left unchanged.

In [7]:
def modified_challenge_attempt(i):
    _, payload = honest_capture()
    p = json.loads(payload)
    p["challenge"]["nonce"] = os.urandom(16).hex()
    p["challenge"]["issued_at"] = int(time.time())
    return server.verify_capture(json.dumps(p), OBS_IP, OBS_UA)

modified_challenge = run_attack("Modified metadata", modified_challenge_attempt)

Modified metadata      attempts=500 accepted=0 (1.6s)  reasons={'challenge does not match server record': 500}


## Results summary (2,500 MITM attempts)

In [8]:
attacks = [replay, stale, nonce, modified_image, modified_challenge]
df_results = pd.DataFrame(attacks)
print(df_results[["Attack Type", "Attempts", "Accepted", "Rejected", "Success Rate (%)"]].to_string(index=False))
total_attempts = int(df_results["Attempts"].sum()); total_accepted = int(df_results["Accepted"].sum())
print(f"\nTotal Attacks: {total_attempts}\nTotal Accepted: {total_accepted}\n"
      f"Overall Success Rate: {100.0*total_accepted/total_attempts:.2f}%")
print(f"Valid control: {control['Accepted']}/{control['Attempts']} accepted")
print("\nRejection reasons:")
for a in attacks:
    print(f"  {a['Attack Type']:<22} {a['Rejection reasons']}")

      Attack Type  Attempts  Accepted  Rejected  Success Rate (%)
           Replay       500         0       500               0.0
  Stale timestamp       500         0       500               0.0
      Nonce reuse       500         0       500               0.0
  Modified binary       500         0       500               0.0
Modified metadata       500         0       500               0.0

Total Attacks: 2500
Total Accepted: 0
Overall Success Rate: 0.00%
Valid control: 100/100 accepted

Rejection reasons:
  Replay                 {'challenge already used (replay)': 500}
  Stale timestamp        {'challenge expired': 500}
  Nonce reuse            {'challenge already used (replay)': 250, 'unknown challenge (not issued by server)': 250}
  Modified binary        {'invalid signature': 500}
  Modified metadata      {'challenge does not match server record': 500}


## Supplementary: website → server channel (500 attempts each, not part of the 2,500)

Tests the fresh $\mathrm{HMAC}_{K_{\rm site}}(siteID\parallel nonce\parallel timestamp\parallel H(body))$ proof that replaces the static `SHA256(shared_key)`. Each attempt starts from a fresh, legitimately obtained token that has not been redeemed, so only the website-channel defect can cause rejection.

In [9]:
def fresh_redemption_payload():
    _, payload = honest_capture()
    res = server.verify_capture(payload, OBS_IP, OBS_UA)
    return victim.build_redemption_payload(res.data["response_json"])

def website_replay_attempt(i):
    req = website.build_server_request(fresh_redemption_payload(), OBS_IP, OBS_UA)
    assert server.redeem_token(req).ok                       # legitimate redemption (not counted)
    return server.redeem_token(req)                          # verbatim replay of the website request

def website_stale_attempt(i):
    old_site = tp.Website(SITE_KEY, WEBSITE_NAME, tp.load_website_secret(), clock=lambda: time.time() - 300)
    return server.redeem_token(old_site.build_server_request(fresh_redemption_payload(), OBS_IP, OBS_UA))

def website_forged_mac_attempt(i):
    rogue = tp.Website(SITE_KEY, WEBSITE_NAME, os.urandom(32).hex().encode())   # attacker without K_site
    return server.redeem_token(rogue.build_server_request(fresh_redemption_payload(), OBS_IP, OBS_UA))

def website_tampered_body_attempt(i):
    req = json.loads(website.build_server_request(fresh_redemption_payload(), OBS_IP, OBS_UA))
    req["user_ip"] = f"10.0.{i % 250}.{1 + i % 250}"         # MITM rewrites context after the MAC
    return server.redeem_token(json.dumps(req))

web_results = [
    run_attack("Website-request replay", website_replay_attempt),
    run_attack("Stale website timestamp", website_stale_attempt),
    run_attack("Forged website MAC", website_forged_mac_attempt),
    run_attack("Tampered request body", website_tampered_body_attempt),
]
print(pd.DataFrame(web_results)[["Attack Type", "Attempts", "Accepted", "Success Rate (%)"]].to_string(index=False))
server.close()

Website-request replay attempts=500 accepted=0 (3.1s)  reasons={'website request replayed': 500}


Stale website timestamp attempts=500 accepted=0 (2.6s)  reasons={'website request expired': 500}


Forged website MAC     attempts=500 accepted=0 (2.6s)  reasons={'invalid website proof': 500}


Tampered request body  attempts=500 accepted=0 (2.6s)  reasons={'invalid website proof': 500}
            Attack Type  Attempts  Accepted  Success Rate (%)
 Website-request replay       500         0               0.0
Stale website timestamp       500         0               0.0
     Forged website MAC       500         0               0.0
  Tampered request body       500         0               0.0
